<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day07-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 7 — In-class discussion problem (3 of 3)

Discuss **as a group first**, write down a guess, then run the code to
check it before presenting.

## Which chain-chain interface actually changes the most?

The main notebook measured *one* distance — the beta1-beta2 heme
iron-to-iron distance — and found it shrinks by about 5 Å going from the
deoxy (T) to the oxy (R) state. But hemoglobin's tetramer has **four**
chain-chain contacts in total: alpha1-beta1, alpha1-beta2, alpha1-alpha2,
and beta1-beta2 (beta1-alpha2 is symmetric to alpha1-beta2, and
alpha2-beta2 is symmetric to alpha1-beta1, so there are only four
genuinely distinct pairs).

Structural biologists (Perutz, and later Baldwin & Chothia, 1979) call
alpha1-beta1 the "fixed"/rigid interface and alpha1-beta2 the "sliding"
interface — the one that rotates the most during the allosteric
transition.

**As a group, before running anything:** given that description, guess
which of the four pairwise Fe-Fe distances you'd expect to change the
**most** between the T-state (`2HHB`) and R-state (`1HHO`) structures,
and which to change the **least**. Then run the cell below, which
computes all four distances in both states (getting the R-state's second
alpha/beta chain the same way the main notebook got its second beta
chain: applying the file's own `REMARK 350` rigid-body symmetry
operation).

In [1]:
import numpy as np
from Bio.PDB import PDBList, PDBParser

pdbl = PDBList()
paths = {
    "2hhb": pdbl.retrieve_pdb_file("2hhb", pdir=".", file_format="pdb"),
    "1hho": pdbl.retrieve_pdb_file("1hho", pdir=".", file_format="pdb"),
}

parser = PDBParser(QUIET=True)
structures = {pdb_id: parser.get_structure(pdb_id, paths[pdb_id]) for pdb_id in paths}

# --- T-state (2HHB): all four chains already present in the file ---
fe_T = {}
for chain in structures["2hhb"][0]:
    for res in chain:
        if res.get_resname() == "HEM":
            fe_T[chain.id] = np.array(res["FE"].get_coord(), dtype=float)
# A, C = alpha chains; B, D = beta chains

# --- R-state (1HHO): only one alpha/beta pair in the file; the other
#     alpha/beta pair is generated by the file's own BIOMT symmetry
#     operation (swap x/y, negate z, no translation) ---
fe_R = {}
for chain in structures["1hho"][0]:
    for res in chain:
        if res.get_resname() == "HEM":
            fe_R[chain.id] = np.array(res["FE"].get_coord(), dtype=float)
Rmat = np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]])
fe_R["Ap"] = Rmat @ fe_R["A"]
fe_R["Bp"] = Rmat @ fe_R["B"]
# A, Ap = alpha chains; B, Bp = beta chains

def dist(fe, a, b):
    return np.linalg.norm(fe[a] - fe[b])

pairs_T = {
    "alpha1-beta1": (dist(fe_T, "A", "B") + dist(fe_T, "C", "D")) / 2,
    "alpha1-beta2": (dist(fe_T, "A", "D") + dist(fe_T, "C", "B")) / 2,
    "alpha1-alpha2": dist(fe_T, "A", "C"),
    "beta1-beta2": dist(fe_T, "B", "D"),
}
pairs_R = {
    "alpha1-beta1": dist(fe_R, "A", "B"),
    "alpha1-beta2": dist(fe_R, "A", "Bp"),
    "alpha1-alpha2": dist(fe_R, "A", "Ap"),
    "beta1-beta2": dist(fe_R, "B", "Bp"),
}

print(f"{'interface':16s} {'T-state':>10s} {'R-state':>10s} {'change':>10s}")
for name in pairs_T:
    t, r = pairs_T[name], pairs_R[name]
    print(f"{name:16s} {t:9.2f}A {r:9.2f}A {t - r:+9.2f}A")

Structure exists: './pdb2hhb.ent' 
Structure exists: './pdb1hho.ent' 
interface           T-state    R-state     change
alpha1-beta1         36.46A     34.73A     +1.73A
alpha1-beta2         24.30A     25.50A     -1.19A
alpha1-alpha2        34.16A     34.75A     -0.59A
beta1-beta2          39.51A     34.62A     +4.89A


**Discussion point.** The raw iron-to-iron distances tell a more
surprising story than "the sliding interface changes the most":

- **beta1-beta2** changes the *most* in raw distance (~4.9 Å) — even
  though Perutz's classic description names alpha1-beta2 as *the*
  "sliding" interface.
- **alpha1-alpha2** barely changes at all (~0.6 Å).
- **alpha1-beta2**, the actual "sliding" interface, changes only
  modestly in pure Fe-Fe distance (~1.2 Å) — despite undergoing the
  largest *rotation* of any interface during the T→R transition.

The resolution: **a single point-to-point distance and a rigid-body
rotation are not the same thing.** Two chains can rotate substantially
relative to each other around a contact point without their heme irons
(which don't have to sit anywhere near that contact point) moving very
far apart — and two chains can move apart in straight-line distance
without rotating much at all. This is exactly why the main book page
mentions that comparing structures properly needs a real rigid-body
transformation (a rotation matrix, not just one distance) to describe
how one part moved relative to another — a single measured distance,
however real and correctly computed, doesn't always tell the whole
story.